## 1. Importation et Exploration des Données

In [ ]:
import pandas as pd
# Importation du jeu données
df = pd.read_csv("../../../DataSets/07/Credit_card_dataset.csv")

df.head(10)

In [ ]:
## La taille de nos donnés
df.shape

In [ ]:
# Resumer statistique des données
df.describe()

In [ ]:
df.info()

## 2. Préparation des Données

### a. Gestion des Valeurs Manquantes

In [ ]:
cols_numeric = df.select_dtypes(include='number').columns
cols_numeric

In [ ]:
# Vérification des valeurs manquantes
print(df.isnull().sum())

# Imputation ou suppresion des valeurs manquantes
df[cols_numeric] = df[cols_numeric].fillna(df[cols_numeric].median())

In [ ]:
df

### b. Gestion des Valeurs Aberrantes

In [ ]:
# Détection des valeurs aberrantes
from scipy.stats import zscore
import numpy as np

z_scores = np.abs(zscore((df[cols_numeric])))

In [ ]:
print(z_scores)

In [ ]:
# Définir un seuil pour les valeurs aberantes (3 est un seuil commun)
threshold = 3

# Identification des indices des observations sans valeurs aberanntes
outliers = (z_scores < threshold).all(axis=1)

In [ ]:
df_clean = df[outliers]

In [ ]:
df_clean.shape

In [ ]:
print(f"Nombre d'observations aberrantes: {df.shape[0] - df_clean.shape[0]}")

## 3. Division des donnée

In [ ]:
from sklearn.model_selection import train_test_split

## Feactures engineering / Selection des caractéristiques
X = df_clean[cols_numeric]

print(f"Taille Train: {X.shape[0]}")

In [ ]:
X.isnull().sum()

### Normalisation

## 3. Clustering Hiérarchique

In [ ]:
# Appliquer le clustering agglomeratif
from sklearn.cluster import AgglomerativeClustering

K = 5
model = AgglomerativeClustering(n_clusters=K, metric='euclidean', linkage='complete')
agg_clustering = model.fit(X)

# Ajouter les labels des clusters au DataFrame d'entrainement
X.loc[:, 'Cluster'] = agg_clustering.labels_

## Visualisation AgglomerativeClustering

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
## Visualisation des clusters
plt.figure(figsize=(10, 7))
plt.scatter(X['PURCHASES'], X['CREDIT_LIMIT'], c=agg_clustering.labels_, cmap='rainbow')
plt.title("AgglomerativeClustering Clustering")
plt.xlabel("Purchases")
plt.ylabel("Credit Limit")
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

# Calcul du score de silhouette pour les données d'entraînement
silhouette_avg = silhouette_score(X[['PURCHASES', 'CREDIT_LIMIT']], agg_clustering.labels_)

print(f"Score moyen de silhouette pour n_clusters = 3 : {silhouette_avg:.3f}")


## Utiliser la Méthode de Silhouette pour Trouver le Nombre Optimal de Clusters

In [ ]:
silhouette_scores = []

# Essayer différents nombres de clusters
for n_clusters in range(2, 11):
    agg_clustering = AgglomerativeClustering(n_clusters=n_clusters, metric='euclidean', linkage='ward')
    cluster_labels = agg_clustering.fit_predict(X[['PURCHASES', 'CREDIT_LIMIT']])

    silhouette_avg = silhouette_score(X[['PURCHASES', 'CREDIT_LIMIT']], cluster_labels)
    silhouette_scores.append(silhouette_avg)
    print(f"Pour n_clusters = {n_clusters}, le score moyen de silhouette est {silhouette_avg:.3f}")

# Tracer les scores de silhouette pour chaque nombre de clusters
plt.figure(figsize=(10, 6))
plt.plot(range(2, 11), silhouette_scores, marker='o')
plt.title("Méthode de Silhouette pour Choisir le Nombre Optimal de Clusters")
plt.xlabel("Nombre de Clusters")
plt.ylabel("Score Moyen de Silhouette")
plt.show()


## 4. Clustering K-means

In [ ]:
X_Kmeans = X[['PURCHASES', 'CREDIT_LIMIT']]

In [ ]:
from sklearn.cluster import KMeans

K = 3
# Determination du nombre de clusters
kmeans = KMeans(n_clusters=K)
kmeans.fit(X_scaled)

# Ajout des labels
X_Kmeans.loc[:, 'Cluster Kmeans'] = kmeans.labels_

In [ ]:
X_Kmeans

In [ ]:
# Visualisation des 
plt.figure(figsize=(10, 7))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=X_Kmeans['Cluster Kmeans'], cmap='rainbow')
plt.title("KMeans Clustering")
plt.xlabel('ACHATS')
plt.ylabel('limite de crédit')
plt.show()

## 5. Détermination de la Meilleure Valeur de k

In [ ]:
# Calcul de la somme des distances au carré pour différentes valeurs de k
inertia = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

# Tracé de la méthode du coude
plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel('Nombre de clusters')
plt.ylabel('Inertia')
plt.show()
